# MCP Client — Connecting to MCP Servers

## Overview

In the previous notebook we explored how MCP **servers** expose tools. Now we look at the
other side of the protocol: the **client** that connects to those servers, discovers their
tools, and invokes them on behalf of an LLM.

The `MCPClientManager` class in AgentExplorr provides a unified interface for:

- **Registering tools** from one or more MCP servers
- **Listing available tools** so the LLM knows what it can do
- **Calling tools** and returning structured results
- **Formatting tool metadata** for LLM frameworks (LangChain, OpenAI, etc.)

## What You Will Learn

1. How to instantiate and configure `MCPClientManager`
2. The `ToolInfo` and `ToolResult` data classes
3. How a client discovers and invokes tools
4. Error handling when tool calls fail
5. How tool descriptions are formatted for LLM consumption

## Learning Resources

| Resource | Link |
|----------|------|
| MCP Client SDK | https://github.com/modelcontextprotocol/python-sdk |
| JSON-RPC 2.0 Spec | https://www.jsonrpc.org/specification |
| Video: MCP Clients Explained | https://www.youtube.com/watch?v=kQmXtrmQ5Zg |

In [ ]:
# ── Import the client module and inspect the core classes ──────────────────
from agentexplorr.mcp.clients.mcp_client import MCPClientManager, ToolInfo, ToolResult

# MCPClientManager is the main entry point.
# It maintains a registry of tools and their handlers.
manager = MCPClientManager()

print("MCPClientManager created.")
print()

# ToolInfo describes a single tool's metadata — what the LLM sees.
print("ToolInfo fields:")
sample_tool = ToolInfo(
    name="read_file",
    description="Read the contents of a file.",
    input_schema={
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "Relative path to the file"}
        },
        "required": ["path"],
    },
)
print(f"  name         : {sample_tool.name}")
print(f"  description  : {sample_tool.description}")
print(f"  input_schema : {sample_tool.input_schema}")
print()

# ToolResult wraps the output of a tool call.
print("ToolResult fields:")
sample_result = ToolResult(content="Hello, world!", is_error=False)
print(f"  content  : {sample_result.content}")
print(f"  is_error : {sample_result.is_error}")

## Client-Server Architecture

MCP follows a classic **client-server** pattern over JSON-RPC 2.0:

```
┌─────────────────┐                        ┌──────────────────┐
│   LLM / Agent   │                        │   MCP Server(s)  │
│                 │                        │                  │
│  "I need to     │   ┌───────────────┐    │  filesystem      │
│   read a file"  │──>│  MCP Client   │───>│  database        │
│                 │<──│  Manager      │<───│  api             │
│  "Got the       │   └───────────────┘    │                  │
│   contents!"    │     JSON-RPC 2.0       │  @server.tool()  │
└─────────────────┘     over stdio/HTTP    └──────────────────┘
```

### The communication flow

1. **Discovery** — The client asks the server: "What tools do you have?"
   The server responds with a list of `ToolInfo` objects (name, description, schema).

2. **Selection** — The LLM reads the tool descriptions and decides which tool to call
   and what arguments to pass.

3. **Invocation** — The client sends a `call_tool` request with the tool name and arguments.
   The server executes the function and returns a `ToolResult`.

4. **Response** — The client passes the result back to the LLM, which incorporates it
   into its response to the user.

### Transport options

| Transport | Use Case | How It Works |
|-----------|----------|--------------|
| **stdio** | Local tools, CLI | Client spawns server as subprocess; JSON over stdin/stdout |
| **SSE** | Web apps, remote | HTTP with Server-Sent Events for streaming |
| **Streamable HTTP** | Modern deployments | Standard HTTP with streaming support |

The `MCPClientManager` abstracts away the transport — you register tools and call them
regardless of whether the server is local or remote.

In [ ]:
# ── Discovering and invoking tools via MCPClientManager ───────────────────
# In a real deployment, tools are discovered automatically when the client
# connects to a server over stdio or HTTP.  Here we register tools manually
# to demonstrate the same workflow without needing a running server process.

import json

# --- Register tools that mirror what the filesystem server exposes ---

manager.register_tool(
    ToolInfo(
        name="read_file",
        description="Read the contents of a file.",
        input_schema={
            "type": "object",
            "properties": {"path": {"type": "string"}},
            "required": ["path"],
        },
    ),
    handler=lambda path: f"(simulated) Contents of {path}",
)

manager.register_tool(
    ToolInfo(
        name="list_directory",
        description="List files and directories at the given path.",
        input_schema={
            "type": "object",
            "properties": {"path": {"type": "string"}},
        },
    ),
    handler=lambda path=".": f"(simulated) Listing of {path}/",
)

manager.register_tool(
    ToolInfo(
        name="run_query",
        description="Run a read-only SQL query and return results.",
        input_schema={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    ),
    handler=lambda query: f"(simulated) Results for: {query}",
)

# --- Discover tools (what the LLM sees) ---
print("=== Registered tools ===")
for tool in manager.list_tools():
    print(f"  {tool.name:20s} — {tool.description}")

print()

# --- Call a tool (what happens when the LLM decides to use one) ---
print("=== Calling 'read_file' with path='README.md' ===")
result = manager.call_tool("read_file", {"path": "README.md"})
print(f"  content  : {result.content}")
print(f"  is_error : {result.is_error}")

print()

# --- Format tools for LLM frameworks ---
print("=== Tools formatted for LLM consumption (get_tools_for_llm) ===")
llm_tools = manager.get_tools_for_llm()
print(json.dumps(llm_tools, indent=2))

## Error Handling and Resilience

Robust error handling is critical in MCP because tool calls can fail for many reasons:
a file might not exist, a database query might have a syntax error, or a remote API
might be unreachable.

### How `MCPClientManager.call_tool()` handles errors

The `call_tool` method in AgentExplorr wraps every handler invocation in a `try/except`
block and returns a `ToolResult` with `is_error=True` when something goes wrong:

```python
def call_tool(self, name: str, arguments: dict) -> ToolResult:
    # 1. Unknown tool name → immediate error
    if name not in self._tools:
        return ToolResult(content=f"Error: Unknown tool '{name}'", is_error=True)

    # 2. No handler registered → error
    handler = self._tool_handlers.get(name)
    if handler is None:
        return ToolResult(content=f"Error: No handler for '{name}'", is_error=True)

    # 3. Handler raises an exception → caught and wrapped
    try:
        result = handler(**arguments)
        return ToolResult(content=str(result))
    except Exception as e:
        return ToolResult(content=f"Error calling {name}: {e}", is_error=True)
```

### Why this matters for LLM agents

When the LLM receives a `ToolResult` with `is_error=True`, it can:
- **Retry** with corrected arguments (e.g., fix a typo in a file path)
- **Fall back** to a different tool (e.g., use `list_directory` before `read_file`)
- **Report the error** to the user with an explanation

This error-aware design lets agents recover gracefully instead of crashing.

## Key Takeaways

1. **`MCPClientManager` is the single entry point.** It maintains a registry of tools
   and their handlers, regardless of whether those tools come from a local server,
   a remote server, or are registered manually.

2. **`ToolInfo` describes *what*; handlers implement *how*.** The `ToolInfo` dataclass
   carries the metadata the LLM needs (name, description, JSON Schema), while the
   handler is the actual callable that executes the tool logic.

3. **`ToolResult` is always returned.** Every `call_tool` invocation returns a
   `ToolResult` with a `content` string and an `is_error` flag -- never a raw
   exception. This gives the LLM a consistent interface to reason about success
   and failure.

4. **`get_tools_for_llm()` bridges MCP and LLM frameworks.** It formats the
   registered tools into the `[{"name", "description", "parameters"}]` structure
   expected by OpenAI, LangChain, and similar tool-calling APIs.

5. **Error handling is built in.** Unknown tool names, missing handlers, and handler
   exceptions are all caught and surfaced as error results, enabling the LLM agent
   to retry or fall back gracefully.

## Next Steps

- **Build a full agent loop** — combine `MCPClientManager` with an LLM API to create
  an agent that discovers tools, decides which to call, and uses the results.
- **Connect to a live server** — start a filesystem or database server process and
  connect to it over stdio using the MCP Python SDK's `ClientSession`.
- **Add your own tools** — create a new MCP server for a service you use (e.g.,
  a calendar API, a CI/CD system, or a monitoring dashboard).